In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from utils import get_data_path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import plotly.graph_objects as go
from xgboost import XGBRegressor

# Setup Paths
notebook_dir = os.path.dirname(os.getcwd())  
project_root = os.path.dirname(notebook_dir) 
sys.path.append(project_root)  

# Load Data
df = pd.read_csv(get_data_path('data/data_raw.csv'))
df['StartDateTime'] = pd.to_datetime(df['StartDateTime'], format='%d/%m/%Y %H:%M')
df.set_index('StartDateTime', inplace=True)
df = df.asfreq('30min')  # setting frequency 

# NaNs 
print("Number of NaNs in data:", df['ISEM DA Price'].isna().sum())

#### Data prep and setting features

In [ ]:
# Load Data
df = pd.read_csv(get_data_path('data/data_raw.csv'))
df['StartDateTime'] = pd.to_datetime(df['StartDateTime'], format='%d/%m/%Y %H:%M')
df.set_index('StartDateTime', inplace=True)
df = df.asfreq('30min')  # Ensure frequency is set

cutoff_date = df.index[-1] - pd.DateOffset(months=12)

df_recent = df[df.index >= cutoff_date].copy()
df_recent = df_recent.sort_index()
time_delta = df_recent.index[1] - df_recent.index[0]
print(f"Time delta between points: {time_delta}")

# external features
external_features = ['DemandForecast-DAM', 'WindForecast-DAM', 'Fuel.Gas', 'Fuel.Carbon']

# time features
df_recent['hour'] = df_recent.index.hour
df_recent['day_of_week'] = df_recent.index.dayofweek
df_recent['month'] = df_recent.index.month
df_recent['is_weekend'] = df_recent.index.dayofweek >= 5
df_recent['quarter_of_day'] = df_recent.index.hour // 6

# price lag features - adapt shifts based on data frequency
points_per_day = 48 if time_delta.total_seconds() == 1800 else 24  # 48 points for 30-min data, 24 for hourly
df_recent['Price_Lag1'] = df_recent['ISEM DA Price'].shift(1)  # Previous period
df_recent['Price_Lag_SameHourYesterday'] = df_recent['ISEM DA Price'].shift(points_per_day)  # same hour yesterday
df_recent['Price_Lag_SameHourLastWeek'] = df_recent['ISEM DA Price'].shift(points_per_day * 7)  # same hour last week

# more features
df_recent['Gas_to_Wind_Ratio'] = df_recent['Fuel.Gas'] / df_recent['WindForecast-DAM'].clip(lower=0.1)
df_recent['Peak_Hour'] = ((df_recent.index.hour >= 7) & (df_recent.index.hour <= 22)).astype(int)

# defione features
features = external_features + [
    'hour', 'day_of_week', 'month', 'is_weekend', 'quarter_of_day', 'Peak_Hour',
    'Price_Lag1', 'Price_Lag_SameHourYesterday', 'Price_Lag_SameHourLastWeek',
    'Gas_to_Wind_Ratio'
]


#### train/test splot, training + preds

In [ ]:
# Train/test split
train_size = int(len(df_recent) * 0.8)
train = df_recent.iloc[:train_size]
test = df_recent.iloc[train_size:]
print(f"Training data: {len(train)} points ({train.index.min()} to {train.index.max()})")
print(f"Testing data: {len(test)} points ({test.index.min()} to {test.index.max()})")

# Prepare data matrices
X_train = train[features]
y_train = train['ISEM DA Price']
X_test = test[features]
y_test = test['ISEM DA Price']

# Define and train model
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=2,
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=42
)

print("Training XGBoost model...")
xgb_model.fit(X_train, y_train)

# Generate predictions
train_preds = xgb_model.predict(X_train)
test_preds = xgb_model.predict(X_test)

# Calculate metrics for test data (out-of-sample)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
mae = mean_absolute_error(y_test, test_preds)
smape_val = smape(y_test, test_preds)
r2 = r2_score(y_test, test_preds)

print("\nXGBoost Model Performance (Test Set):")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"sMAPE: {smape_val:.2f}%")
print(f"R² Score: {r2:.4f}")

# Calculate metrics for negative prices specifically
neg_price_idx = y_test < 0
if sum(neg_price_idx) > 0:
    neg_price_rmse = np.sqrt(mean_squared_error(y_test[neg_price_idx], test_preds[neg_price_idx]))
    print(f"RMSE on negative prices: {neg_price_rmse:.2f}")

# Feature importance analysis
importance = pd.DataFrame({
    'Feature': features,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 5 most important features:")
print(importance.head(5))


In [ ]:
train_preds_series = pd.Series(train_preds, index=train.index)
test_preds_series = pd.Series(test_preds, index=test.index)

# Create plot
fig = go.Figure()

# Training data
fig.add_trace(go.Scatter(
    x=train.index,
    y=train['ISEM DA Price'],
    name='Training Data',
    line=dict(color='blue')
))

# Test data
fig.add_trace(go.Scatter(
    x=test.index,
    y=test['ISEM DA Price'],
    name=' Test Values',
    line=dict(color='green')
))

# Out-of-sample forecast
fig.add_trace(go.Scatter(
    x=test.index,
    y=test_preds_series,
    name='XGBoost Forecast',
    line=dict(color='red', dash='dash')
))

# Add vertical line for train/test split
split_date = test.index[0]
fig.add_vline(x=split_date, line_width=2, line_dash="dash", line_color="black")
fig.add_annotation(
    x=split_date,
    y=df_recent['ISEM DA Price'].max() * 0.9,
    text="Train-Test Split",
    showarrow=True,
    arrowhead=1,
    ax=-60,
    ay=-40
)

# Layout
fig.update_layout(
    title="XGBoost Electricity Price Forecast",
    xaxis_title="Date",
    yaxis_title="Price (€/MWh)",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.1)
)

# Range selector - FIX: Change "week" to "day" with count=7
fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=3, label="3m", step="month", stepmode="backward"),
            dict(count=7, label="1w", step="day", stepmode="backward"),  # Changed from "week" to "day" with count=7
            dict(step="all")
        ])
    )
)

# Show plot
fig.show()